In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd /content/drive/MyDrive/IntroAI/Lab7

# Lab 7 Dataset

This part prepares data for developing a convolutional neural network (CNN) using PyTorch for solving a simple classification problem using the MNIST dataset of fashion. It should be used in combination with a CNN model, such as a self-defined CNN, AlexNet, ResNet or a VGGNet.


You need to import the package `torch` in order to use PyTorch and `torchvision` to use methods of dealing with datasets. If you don't have these packages in your environment, you need to install them first using `pip install <package-name>`.

You also need a set of predefined classes which are packed in `heading.py`, such as AlexNet, ResNet and VGG.

In [ ]:
from heading import *
import torch
from torchvision import datasets

### THE MNIST DATABASE OF FASHIONS

Fashion-MNIST is a dataset of Zalando's article images—consisting of a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes.
Link: https://www.kaggle.com/zalando-research/fashionmnist

Labels

Each training and test example is assigned to one of the following labels:

    0 T-shirt/top
    1 Trouser
    2 Pullover
    3 Dress
    4 Coat
    5 Sandal
    6 Shirt
    7 Sneaker
    8 Bag
    9 Ankle boot

It is a good database for people who want to learn image classification methods using real-world data.

Since the input image size is different in different CNN architectures, you need to prepare datasets based on each CNN architecture.

You download the datasets and store them in a sub-folder "Data" in your Lab 7 folder.

### Load MNIST datasets

First you specify the input image size used in a target CNN model.
* If you use the self-defined CNN or the ResNet, the image dimension is 28 * 28, so the image size is 28.
* If you use the AlexNet, the image dimension is 227 * 227, so the image size is 227.
* If you use the VGGNet, the image dimension is 224 * 224, so the image size is 224.

In [ ]:
image_size = 28

Then, you need to tranform the original image shape to the image size used in the target network by using `resize_image(<image-size>)` method.

In [ ]:
transform = resize_image(image_size)

Divide the dataset into BATCH_SIZE equal parts for training.

## Important message

A CNN variant, such as AlexNet, ResNet or VGGNet, is usually a large scale network. If you train it using the entire dataset, it might cost several hours (on cpu). This lab is mainly for you to learn how to configure and how to train a CNN model, thus, you can set the batch size to be 1 for simplicity. If you want to try with the entire dataset, you can set the batch size to 64.

In [ ]:
BATCH_SIZE = 1

You need to define a `load_data()` method to load training dataset and test dataset based on the BATCH_SIZE.

Here you define this method for loading the dataset Fashion-MNIST. If you want to try other datasets, you can change the dataset name and path to accommodate your new dataset, e.g. `datasets.MNIST`, `datasets.CIFAR10`, or `datasets.CIFAR10`.

In [ ]:
def load_data(transform, BATCH_SIZE, train, PATH = 'Data'):
    load_dataset = datasets.FashionMNIST(PATH,
                               train=train,
                               transform=transform,
                               download=False)

    data_loader = torch.utils.data.DataLoader(dataset = load_dataset,
                                            batch_size = BATCH_SIZE, shuffle = True)

    return data_loader

You then load the training data and test data separately using the `load_data()` method. For the training data, you need to set the argument `train` to be `True`. For the testing data, you need to set the this argument to be `False`. The path is `Data` which is in the current directory.

In [ ]:
train_loader = load_data(transform, BATCH_SIZE, train = True, PATH = 'Data')
test_loader = load_data(transform, BATCH_SIZE, train = False, PATH = 'Data')

You need to convert the labels to object's names. You can create a list that contains all categories in order.

In [ ]:
label_list = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

You can make a future image sample set. You can use the CNN model to predict the labels of these samples.
In order to do that, you need to specify the location of your future image data by using the`FlameSet(image-path, image-size)`, where `image-path` is the path to the folder containing the images and image-size is the image size used in the pre-trained model. In this case, the folder path is Image in the current directory, i.e., ./Image.
The image size should match with the target CNN model.
* If you use the AlexNet, the image dimension is 227 * 227, so the image size is 227.
* If you use the ResNet, the image dimension is 28 * 28, so the image size is 28.
* If you use the VGGNet, the image dimension is 224 * 224, so the image size is 224.


In [ ]:
dataSet = FlameSet('./Image', image_size = 28)

You can check the dataset specified by listing the files in this linked folder.

In [ ]:
dataSet.get_image_list()

Once the image files are confirmed, you can select one image or all to predict the output label(s) using the target CNN model.

In [ ]:
# Select the 4th image file in the specified folder.
select_image = dataSet[3]

### View MNIST datasets

You can view data samples in a batch in the train or test or future sample loader, i.e., you can view BATCH_SIZE images and their labels at each time.  For example, when BATCH_SIZE = 1, you can view the image in the train or test loader as a tensor that represents the image as a 227x227 pixel matrix.

In [ ]:
view_datasets(train_loader, label_list)

In [ ]:
view_datasets(test_loader, label_list)

You can also view your selected image sample using methods in OpenCV.

In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
img=cv.imread('./Image/5-0.png')
cv2_imshow(img)

# Lab 7 Algorithm -- Self-defined CNN
In this part, you will learn how to use PyTorch to train a self-defined convolutional neural network for solving a given problem using the dataset sepcified above.

You will develop a CNN model from scratch. You first configure this convolutional neural network, then train and test it, lastly apply it to classify future images.  

#### Input channel:
* Input channel number is 1 for gray images (3 for RGB colour images)

#### Two convolutional layers: CNV_layer 1 and CNV_layer 2
* CNV_layer 1:  
 * convolutional layer parameters: input channels:1, output channels:16, filter size: 3x3, stride: 2, and padding: 1
 * max pooling layer: 2x2
 * activation function: ReLU()
 * Batch normalisation: 16

* CNV_layer 2:
 * convolutional layern parameters: input channels:16, output channels:32, filter size: 3x3, stride: 1, and padding: 1
 * Max pooling layer:2x2
 * Activation function: ReLU()
 * Batch normalisation: 32

#### Flattening
The output channels of CNV_layer 2 will be flattened to be a column vector of size 288 (.

#### Three fully connected layers: FC_layer_1,  FC_layer_2 and FC_layer_3
 * FC_layer_1 parameters: Input nodes: 288; Output nodes: 60; Drop out ratio: 0.5; Activation function: ReLU()
 * FC_layer_2 parameters: Input nodes: 60; Output nodes: 20; Drop out ratio: 0.5; Activation function: ReLU()
 * FC_layer_3 parameters: Input nodes: 20; Output nodes: 10; Activation function: Softmax()
  
#### Output
  * output nodes: 10 for 10 classes with each class representing one digit.

First, you can define the CNN class with the structural parameters above.

In [ ]:
class CONV_net(torch.nn.Module):
    def __init__(self, input_channels = 1, output_size = 10):
        super().__init__()
        self.conv1 = torch.nn.Sequential(
            torch.nn.Conv2d(input_channels,16,3,1,1),
            torch.nn.MaxPool2d(2),
            torch.nn.ReLU(),
            torch.nn.BatchNorm2d(16)
        )
        self.conv2 = torch.nn.Sequential(
            torch.nn.Conv2d(16,32,3,2,1),
            torch.nn.MaxPool2d(2),
            torch.nn.ReLU(),
            torch.nn.BatchNorm2d(32)
        )
        self.fc1 = torch.nn.Sequential(
            torch.nn.Linear(288,60),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU()
        )
        self.fc2 = torch.nn.Sequential(
            torch.nn.Linear(60,20),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU()
        )
        self.fc3 = torch.nn.Sequential(
            torch.nn.Linear(20, output_size),
            torch.nn.Softmax()
        )


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        print(x.shape)  # Add this line
        x = x.view(-1,288)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

Then, you create an instance of the class of CONV_net above by passing in two arguments, input_channels and outout_size. input_channels = 1 is for gray images and input_channels = 3 for colour images. output size is the number of classes. These two arguments are problem specific.

For image classification using the MNIST dataset of Fashion, input_channels = 1 for gray images and output_size = 10 for 10 classes of Fashion items to be classified.

In [ ]:
input_channels = 1
output_size = 10

In [ ]:
convlution = CONV_net(input_channels, output_size)

Followed, you can build a convolutional neural network by using `create_network(<CONV_net instance>)`.

In [ ]:
net_work = create_network(convlution)

You need to prepare parameters for training, which are

* the `number of epoches` and
* the `learning rate`.

You can set the `number of epoches` as 1 for simplicity and the `learning rate` as 0.001.

In [ ]:
EPOCH = 1
LR = 0.001

Now, you can train the model by using the method `train_model(target-Network,train-loader, learning-rate, number-of-epoches)`.

In [ ]:
trained_model = train_model(net_work, train_loader, LR, EPOCH)

After training, you can test the model using the test dataset by using the method `test_model(trained-model, test-loader)`. Without specifying the number of images used, you use all the images from the test dataset.

In [ ]:
test_model(trained_model, test_loader)

Lastly, you can use the method `predict_image( trained-model, image-data, objective-list, number-of-prediction)` to classify your selected image sample, which is defined in the dataset part.

In [ ]:
predict_image(trained_model, select_image, label_list)